# Experiments

This notebook contains the experimental setup for the interaction between RM3 query expansion and TAS-B neural re-ranking.

Planned pipelines:
- BM25
- BM25 + RM3
- BM25 + TAS-B
- BM25 + RM3 + TAS-B


## Setup

Import PyTerrier and initialize the Java backend.

In [2]:
import pyterrier as pt
import pandas as pd

if not pt.java.started():
    pt.java.init()

## Load dataset

Load the ANTIQUE dataset, topics, and relevance judgments.

In [4]:
dataset = pt.get_dataset("vaswani")
topics = dataset.get_topics()
qrels = dataset.get_qrels()

print(f"Number of topics: {len(topics)}")
print(f"Number of qrels: {len(qrels)}")

topics.head()

Number of topics: 93
Number of qrels: 2083


,qid,query
0,1,measurement of dielectric constant of liquids ...
1,2,mathematical analysis and design details of wa...
2,3,use of digital computers in the design of band...
3,4,systems of data coding for information transfer
4,5,use of programs in engineering testing of comp...


In [5]:
qrels.head()

,qid,docno,label
0,1,1239,1
1,1,1502,1
2,1,4462,1
3,1,4569,1
4,1,5472,1


## Build / load index

Use the course notebook code here to either create an index or load an existing one.

In [10]:
from pathlib import Path

idx_path = Path.cwd().parent / "results" / "vaswani_index"

if not idx_path.exists():
    print("Indexing...")

    indexer = pt.IterDictIndexer(
        str(idx_path),
        meta={'docno': 50, 'text': 4096}
    )
    index_ref = indexer.index(dataset.get_corpus_iter())

else:
    print("Loading existing index...")
    index_ref = str(idx_path)

index = pt.IndexFactory.of(index_ref)

Loading existing index...


In [11]:
print(index.getCollectionStatistics().toString())

Number of documents: 11429
Number of terms: 7756
Number of postings: 224573
Number of fields: 1
Number of tokens: 271581
Field names: [text]
Positions:   false



## BM25 baseline

Create the first retrieval pipeline.

In [9]:
bm25 = pt.terrier.Retriever(index, wmodel="BM25")
bm25(topics.head(5))

,qid,docid,docno,rank,score,query
0,1,8171,8172,0,24.566031,measurement of dielectric constant of liquids ...
1,1,9880,9881,1,22.110514,measurement of dielectric constant of liquids ...
2,1,5501,5502,2,21.717148,measurement of dielectric constant of liquids ...
3,1,1501,1502,3,19.478355,measurement of dielectric constant of liquids ...
4,1,9858,9859,4,18.626342,measurement of dielectric constant of liquids ...
...,...,...,...,...,...,...
4717,5,10015,10016,717,2.379234,use of programs in engineering testing of comp...
4718,5,3484,3485,718,2.335306,use of programs in engineering testing of comp...
4719,5,8380,8381,719,2.313945,use of programs in engineering testing of comp...
4720,5,3333,3334,720,2.096265,use of programs in engineering testing of comp...


## Evaluate BM25

Run the baseline and inspect the first metrics.

In [12]:
pt.Experiment(
    [bm25],
    topics,
    qrels,
    eval_metrics=["ndcg_cut_10", "map", "recall"],
    names=["BM25"]
)

,name,ndcg_cut_10,map,R@5,R@10,R@15,R@20,R@30,R@100,R@200,R@500,R@1000
0,BM25,0.446609,0.296517,0.162592,0.218513,0.260196,0.300136,0.374032,0.599046,0.73208,0.867562,0.934607


## Add RM3

Create the query expansion pipeline. Later, this section can be extended to vary `fb_terms`.

In [15]:
rm3 = pt.rewrite.RM3(index, fb_terms=10)
bm25_rm3 = bm25 >> rm3 >> bm25

In [16]:
pt.Experiment(
    [bm25, bm25_rm3],
    topics,
    qrels,
    eval_metrics=["ndcg_cut_10", "map", "recall"],
    names=["BM25", "BM25+RM3"]
)

,name,ndcg_cut_10,map,R@5,R@10,R@15,R@20,R@30,R@100,R@200,R@500,R@1000
0,BM25,0.446609,0.296517,0.162592,0.218513,0.260196,0.300136,0.374032,0.599046,0.732080,0.867562,0.934607
1,BM25+RM3,0.445102,0.298411,0.163868,0.220062,0.257444,0.295142,0.360578,0.605223,0.736376,0.867241,0.938389


In [17]:
results = []

for fb in [5, 10, 20]:
    rm3 = pt.rewrite.RM3(index, fb_terms=fb)
    pipe = bm25 >> rm3 >> bm25

    exp = pt.Experiment(
        [pipe],
        topics,
        qrels,
        eval_metrics=["ndcg_cut_10", "map", "recall"],
        names=[f"BM25+RM3_fb{fb}"]
    )
    results.append(exp)

pd.concat(results)

,name,ndcg_cut_10,map,R@5,R@10,R@15,R@20,R@30,R@100,R@200,R@500,R@1000
0,BM25+RM3_fb5,0.424501,0.284533,0.157064,0.213089,0.247113,0.285374,0.346861,0.592548,0.719203,0.850211,0.930048
0,BM25+RM3_fb10,0.445102,0.298411,0.163868,0.220062,0.257444,0.295142,0.360578,0.605223,0.736376,0.867241,0.938389
0,BM25+RM3_fb20,0.453192,0.305450,0.164860,0.224261,0.267401,0.298835,0.365280,0.617287,0.746320,0.880182,0.943137


## Add TAS-B

Create the neural re-ranking pipeline.

## Full pipeline: BM25 + RM3 + TAS-B

Combine query expansion and neural re-ranking.

## Compare pipelines

Compare all pipelines using the same evaluation setup.

## RM3 parameter study

Use this section later to test different `fb_terms` values.